# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos: sistema LLM + RAG con información tabular**

* **Nombres y matrículas:**




*   Elemento de lista
*   Elemento de lista
*   Elemento de lista





* ##### **El formato de este cuaderno de Jupyter es libre, pero incluye al menos lo solicitado en el archivo PDF asociado a esta actividad.**

* ##### **Pueden importar los paquetes o librerías que requieran.**

* ##### **Pueden incluir las celdas y líneas de código que deseen.**

In [1]:
# !pip install -r ../requirements.txt

In [2]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

from pypdf import PdfReader
import os
from unstructured.partition.pdf import partition_pdf
import nltk
import re

/Users/alonsopedreromartinez/Documents/GitHub/llms-equipo14/activity5/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/alonsopedreromartinez/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/alonsopedreromartinez/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 1. Los archivos python_cheatsheet.pdf y ml_cheatsheet.pdf los encuentras en la carpeta de la actividad enCanvas.

In [4]:
PDF_PATH = "../files"
PYTHON_CHEATSHEET_FILENAME = "python_cheatsheet.pdf"
ML_CHEATSHEET_FILENAME = "ml_cheatsheet.pdf"

In [5]:
python_cheatsheet = os.path.join(PDF_PATH, PYTHON_CHEATSHEET_FILENAME)
ml_cheatsheet = os.path.join(PDF_PATH, ML_CHEATSHEET_FILENAME)

In [44]:
python_cheatsheet_text = ""
reader = PdfReader(python_cheatsheet)

for page in reader.pages:
    python_cheatsheet_text += page.extract_text()

In [7]:
elements = partition_pdf(
    ml_cheatsheet,
    languages=['eng'],
    strategy="hi_res",
    infer_table_structure=True,
    chunking_strategy="by_title",
    max_characters=1200,
    combine_text_under_n_chars=300
)

table_chunks = []
other_chunks = []

for e in elements:
    if type(e).__name__ == "TableChunk":
        table_chunks.append(e.text)
    else:
        other_chunks.append(e.text)

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 7522.33it/s]
OMP: Warning #96: Cannot form a team with 4 threads, using 1 instead.
OMP: Hint Consider unsetting KMP_DEVICE_THREAD_LIMIT (KMP_ALL_THREADS), KMP_TEAMS_THREAD_LIMIT, and OMP_THREAD_LIMIT (if any are set).


In [8]:
elements_ocr = partition_pdf(
    filename=ml_cheatsheet,
    strategy="ocr_only"
)

all_text = [e.text for e in elements] + [e.text for e in elements_ocr]

In [10]:
def clean_np_str(text):
    return re.sub(r"np\.str_\('([^']*)'\)", r"\1", text)

In [11]:
clean_text = []
for i in range(len(all_text)):           
    clean_text.append(clean_np_str(all_text[i]))


In [14]:
algorithms = [
    "Linear Regression",
    "Logistic Regression",
    "Ridge Regression",
    "Lasso Regression",
    "Decision Tree",
    "Random Forests",
    "Gradient Boosting Regression",
    "XGBoost",
    "LightGBM Regressor",
    "K-Means",
    "Hierarchical Clustering",
    "Gaussian Mixture Models",
    "Apriori algorithm"
]

info_headers = ["DESCRIPTION", "APPLICATIONS", "ADVANTAGES", "DISADVANTAGES"]

In [15]:
sections = {header: [] for header in info_headers}
current_header = None
current_block = []

for item in clean_text:
    if item in info_headers:
        # guardar bloque anterior
        if current_header and current_block:
            sections[current_header].append(current_block)

        # iniciar nuevo bloque
        current_header = item
        current_block = []
    else:
        if current_header:
            current_block.append(item)

# guardar último bloque
if current_header and current_block:
    sections[current_header].append(current_block)

print(sections)

{'DESCRIPTION': [['A simple algorithm that models a linear relationship between inputs and a continuous numerical output variable', 'A simple algorithm that models a linear relationship between inputs and a categorical output (1 or O)', 'Part of the regression family — it penalizes features that have low predictive outcomes by shrinking their coefficients closer to zero. Can be used for classification or regression', 'Part of the regression family — it penalizes features that have low predictive outcomes by shrinking their coefficients to zero. Can be used for classification or regression', 'Decision Tree models make decision rules on the features to produce predictions. It can be used for classification or regression', 'An ensemble learning method that combines the output of multiple decision trees', 'Gradient Boosting Regression employs boosting to make predictive models from an ensemble of weak predictive learners', 'Gradient Boosting algorithm that is efficient & flexible. Can be u

In [26]:
applications = sections["APPLICATIONS"][0]

grouped_apps = []

for item in applications:
    item = item.strip()

    item = item.replace("USE CASES", "").strip()

    grouped_apps.append([item])

grouped_clean_applications = []
current_group = []
last_num = 0

for block in grouped_apps:
    for item in block:
        if not item.strip():
            continue

        matches = re.findall(r'(\d+)\.\s*[^0-9]+', item)

        # reconstruimos cada elemento como string separado
        parts = re.split(r'(?=\d+\.\s)', item)

        for p in parts:
            p = p.strip()
            if not p:
                continue

            num = int(re.match(r'(\d+)\.', p).group(1))

            # si reinicia secuencia
            if num < last_num:
                grouped_clean_applications.append(current_group)
                current_group = []

            current_group.append(p)
            last_num = num

if current_group:
    grouped_clean_applications.append(current_group)

print(grouped_clean_applications)

[['1. Stock price prediction', '2. Predicting housing prices', '3. Predicting customer lifetime value'], ['1. Credit risk score prediction', '2. Customer churn prediction'], ['1. Predictive maintenance for automobiles', '2. Sales revenue prediction'], ['1. Predicting housing prices', '2. Predicting clinical outcomes based on health data'], ['1. Customer churn prediction', '2. Credit score modeling', '3. Disease prediction'], ['1. Credit score modeling', '2. Predicting housing prices'], ['1. Predicting car emissions', '2. Predicting ride hailing fare amount'], ['1. Churn prediction', '2. Claims processing in insurance'], ['1. Predicting flight time for airlines', '2. Predicting cholesterol levels based on health data'], ['1. Customer segmentation', '2. Recommendation systems'], ['1. Fraud detection', '2. Document clustering based on similarity'], ['1. Customer segmentation', '2. Recommendation systems'], ['1. Product placements', '2. Recommendation engines', '3. Promotion optimization']

In [34]:
grouped_clean_descriptions = []

for desc in sections["DESCRIPTION"][0]:
    grouped_clean_descriptions.append([desc])

print(grouped_clean_descriptions)

[['A simple algorithm that models a linear relationship between inputs and a continuous numerical output variable'], ['A simple algorithm that models a linear relationship between inputs and a categorical output (1 or O)'], ['Part of the regression family — it penalizes features that have low predictive outcomes by shrinking their coefficients closer to zero. Can be used for classification or regression'], ['Part of the regression family — it penalizes features that have low predictive outcomes by shrinking their coefficients to zero. Can be used for classification or regression'], ['Decision Tree models make decision rules on the features to produce predictions. It can be used for classification or regression'], ['An ensemble learning method that combines the output of multiple decision trees'], ['Gradient Boosting Regression employs boosting to make predictive models from an ensemble of weak predictive learners'], ['Gradient Boosting algorithm that is efficient & flexible. Can be use

In [ ]:
grouped_clean_advantages = []
current_group = []
last_num = 0

for item in sections["ADVANTAGES"][0]:
    if not item.strip():
        continue

    parts = re.split(r'(?=\d+\.\s)', item)

    for p in parts:
        p = p.strip()
        if not p:
            continue

        num = int(re.match(r'(\d+)\.', p).group(1))

        if num < last_num:
            grouped_clean_advantages.append(current_group)
            current_group = []

        current_group.append(p)
        last_num = num

if current_group:
    grouped_clean_advantages.append(current_group)

print(grouped_clean_advantages)

[['1. Explainable method', '2. Interpretable results by its output coefficients', '3. Faster to train than other machine learning models'], ['1. Interpretable and explainable', '2. Less prone to overfitting when using regularization', '3. Applicable for multi-class predictions'], ['1. Less prone to overfitting', '2. Best suited where data suffer from multicollinearity', '3. Explainable & interpretable'], ['1. Less prone to overfitting', '2. Can handle high-dimensional data', '3. No need for feature selection'], ['1. Explainable and interpretable', '2. Can handle missing values'], ['1. Reduces overfitting', '2. Higher accuracy compared to other models'], ['1. Better accuracy compared to other regression models', '2. It can handle multicollinearity', '3. It can handle non-linear relationships'], ['1. Provides accurate results', '2. Captures non linear relationships'], ['1. Can handle large amounts of data', '2. Computational efficient & fast training speed', '3. Low memory usage'], ['1. 

In [41]:
grouped_clean_disadvantages = []
current_group = []
last_num = 0

for item in sections["DISADVANTAGES"][0]:
    if not item.strip():
        continue

    parts = re.split(r'(?=\d+\.\s)', item)

    for p in parts:
        p = p.strip()
        if not p:
            continue

        num = int(re.match(r'(\d+)\.', p).group(1))

        # cortar si reinicia O repite número
        if num <= last_num:
            grouped_clean_disadvantages.append(current_group)
            current_group = []

        current_group.append(p)
        last_num = num

if current_group:
    grouped_clean_disadvantages.append(current_group)

print(grouped_clean_disadvantages)

[['1. Assumes linearity between inputs and output', '2. Sensitive to outliers', '3. Can underfit with small, high-dimensional data'], ['1. Assumes linearity between inputs and outputs', '2. Can overfit with small, high-dimensional data'], ['1. All the predictors are kept in the final model', "2. Doesn't perform feature selection"], ['1. Can lead to poor interpretability as it can keep highly correlated variables'], ['1. Prone to overfitting', '2. Sensitive to outliers'], ['1. Training complexity can be high', '2. Not very interpretable'], ['1. Sensitive to outliers and can therefore cause overfitting', '2. Computationally expensive and has high complexity'], ['1. Hyperparameter tuning can be complex', '2. Does not perform well on sparse datasets'], ['1. Can overfit due to leaf-wise splitting and high sensitivity', '2. Hyperparameter tuning can be complex'], ['1. Requires the expected number of clusters from the beginning', '2. Has troubles with varying cluster sizes and densities'], ["

In [55]:
ml_cheatsheet_text = ""
for i in range(len(algorithms)):
    #algorithms
    ml_cheatsheet_text += "ALGORITHM "
    ml_cheatsheet_text += algorithms[i]
    #descriptions
    ml_cheatsheet_text += (info_headers[0] + " ")
    ml_cheatsheet_text += (str(grouped_clean_descriptions[i]) + " ")
    #applications
    ml_cheatsheet_text += (info_headers[1] + " ")
    ml_cheatsheet_text += (str(grouped_clean_applications[i]) + " ")
    #advantages
    ml_cheatsheet_text += (info_headers[2] + " ")
    ml_cheatsheet_text += (str(grouped_clean_advantages[i]) + " ")
    #disadvantages
    ml_cheatsheet_text += (info_headers[3] + " ")
    ml_cheatsheet_text += (str(grouped_clean_disadvantages[i]) + " ")

print(ml_cheatsheet_text)

ALGORITHM Linear RegressionDESCRIPTION ['A simple algorithm that models a linear relationship between inputs and a continuous numerical output variable'] APPLICATIONS ['1. Stock price prediction', '2. Predicting housing prices', '3. Predicting customer lifetime value'] ADVANTAGES ['1. Explainable method', '2. Interpretable results by its output coefficients', '3. Faster to train than other machine learning models'] DISADVANTAGES ['1. Assumes linearity between inputs and output', '2. Sensitive to outliers', '3. Can underfit with small, high-dimensional data'] ALGORITHM Logistic RegressionDESCRIPTION ['A simple algorithm that models a linear relationship between inputs and a categorical output (1 or O)'] APPLICATIONS ['1. Credit risk score prediction', '2. Customer churn prediction'] ADVANTAGES ['1. Interpretable and explainable', '2. Less prone to overfitting when using regularization', '3. Applicable for multi-class predictions'] DISADVANTAGES ['1. Assumes linearity between inputs and 

## 2. Pueden elegir el modelo de lenguaje de gran tamaño, LLM, que consideren más adecuado para su chatbot. Observa que los archivos están en inglés, por lo que puedes decidir usar un chatbot en español con un LLM multilingüe o manejar toda la solución en idioma inglés.

## 3. El chatbot deberá integrar un sistema de recuperación aumentada generativa (RAG) que permita mejorar la precisión de las respuestas mediante la consulta de dichas hojas de referencia (cheat-sheet).

## 4. Pueden utilizar FAISS o ChromaDB para la construcción de la base de datos vectorial. Igualmente está a tu consideración utilizar Gradio o ejecutarlo directamente en tu cuaderno de Jupyter.

## 5. Una vez implementado el chatbot, responder las siguientes preguntas:

- a. Según la sección de 'Exceptions', ¿qué excepción se lanza si divido entre cero?
- b. ¿Puedes proporcionarme 2 ejemplos de métodos de cadena (string methods)?
- c. ¿Cuáles son las principales desventajas de utilizar el modelo Bosque Aleatorio
(Random Forests)?
- d. ¿Cuáles son 3 casos de uso (use cases) de las técnicas de aprendizaje no
supervisado (Unsupervised Learning) para las técnicas de agrupamiento
(clustering)?
- e. f. Incluye al menos una pregunta más sobre la hoja de Python.
Incluye al menos una pregunta más sobre la hoja de ML.

# **Conclusiones:**

* #### **Incluyan sus conclusiones de la actividad chatbot LLM + RAG para documentos con información tabular:**



None

# **Fin de la actividad chatbot: LLM + RAG**